# Análise por processo da otimização de prompts

Este notebook analisa apenas o processo em `out/prompt_optimization/Llama3.1-I/user_knn/sep/repr_llm2vec/early_false/mmr_lambda_1_0/mmr_pool_10/Llama3.1-I/prompt_opt/sep`.

Fluxo sugerido:
1. conferir o resumo e os artefatos carregados neste diretório de processo;
2. inspecionar o prompt base, o melhor prompt salvo, o melhor prompt em validação e o comportamento das métricas por época;
3. se necessário, ajustar `PROCESS_DIR` para outro diretório de processo compatível.

Observações importantes:
- `best_prompt.json` hoje salva o **melhor prompt de treino** (`best_on_train`).
- Quando o melhor resultado em validação cai em outra época, o notebook destaca essa divergência.
- O padrão de cores é fixo em todos os gráficos: `etd` em azul, `sep` em laranja e `sep_etd_f1` em verde.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

def _is_project_root(candidate: Path) -> bool:
    return (candidate / "run_prompt_optimizer.py").exists() and (candidate / "out").exists()


def find_local_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    descendant_hints = []
    for base in (start, *start.parents):
        descendant_hints.extend(
            [
                base / "prompt-optim-expl-rec" / "explainability-with-LLMs",
                base / "explainability-with-LLMs",
            ]
        )

    for candidate in descendant_hints:
        if _is_project_root(candidate):
            return candidate

    raise FileNotFoundError(
        "Não foi possível localizar a raiz de explainability-with-LLMs. "
        "Execute o notebook no projeto, em um subdiretório dele ou a partir da raiz do workspace."
    )

PROJECT_ROOT = find_local_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.optimization_process_analysis import (
    discover_optimization_processes,
    load_process_bundle,
    summarize_processes,
)

METRIC_COLOR_MAP = {
    "etd": "#1f77b4",
    "sep": "#ff7f0e",
    "sep_etd_f1": "#2ca02c",
    "geom_mean": "#d62728",
    "mean_balance": "#8c564b",
}


def metric_color(metric_name: str | None, default: str = "#4C78A8") -> str:
    return METRIC_COLOR_MAP.get(str(metric_name), default)


METRIC_LABEL_MAP = {
    "etd": "ETD",
    "sep": "SEP",
    "sep_etd_f1": "SEP_ETD_F1",
    "geom_mean": "Geométrica",
    "mean_balance": "Mean Balance",
}


def metric_label(metric_name: str | None) -> str:
    return METRIC_LABEL_MAP.get(str(metric_name), str(metric_name))


def blend_with_white(color: str, blend: float) -> str:
    red, green, blue = mcolors.to_rgb(color)
    return mcolors.to_hex(
        tuple((1 - blend) * channel + blend for channel in (red, green, blue))
    )


def metric_shades(metric_name: str | None, size: int) -> list[str]:
    base_color = metric_color(metric_name)
    if size <= 1:
        return [base_color]

    max_blend = 0.45
    return [
        blend_with_white(base_color, max_blend * (index / max(1, size - 1)))
        for index in range(size)
    ]


def ensure_balance_metrics(epochs_df: pd.DataFrame) -> pd.DataFrame:
    epochs_df = epochs_df.copy()

    for split in ("train", "val"):
        sep_col = f"{split}_score_sep"
        etd_col = f"{split}_score_etd"

        if sep_col not in epochs_df.columns or etd_col not in epochs_df.columns:
            continue

        sep_series = pd.to_numeric(epochs_df[sep_col], errors="coerce")
        etd_series = pd.to_numeric(epochs_df[etd_col], errors="coerce")

        epochs_df[f"{split}_score_geom_mean"] = (
            sep_series.clip(lower=0) * etd_series.clip(lower=0)
        ).pow(0.5)
        epochs_df[f"{split}_score_mean_balance"] = (
            ((sep_series + etd_series) / 2.0)
            * (1 - (sep_series - etd_series).abs())
        )

    return epochs_df


def preferred_metric_order(metric_names: list[str]) -> list[str]:
    preferred = ["sep", "etd", "sep_etd_f1", "geom_mean", "mean_balance"]
    ordered = [metric_name for metric_name in preferred if metric_name in metric_names]
    ordered.extend(metric_name for metric_name in metric_names if metric_name not in ordered)
    return ordered

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 50)

PROJECT_ROOT


In [ ]:
ANALYSIS_ROOT_REL = "out/prompt_optimization/Llama3.1-I/user_knn/sep/repr_llm2vec/early_false/mmr_lambda_1_0/mmr_pool_10/Llama3.1-I/prompt_opt/sep"
ANALYSIS_ROOT = PROJECT_ROOT / ANALYSIS_ROOT_REL
PROCESS_INDEX = 0
PROCESS_DIR = str(ANALYSIS_ROOT)

process_catalog = discover_optimization_processes(PROJECT_ROOT, search_root=ANALYSIS_ROOT)
if process_catalog.empty:
    raise FileNotFoundError(f"Nenhum optimization_process_metadata.json foi encontrado em {ANALYSIS_ROOT}.")

display(Markdown(f"## Processo em `{ANALYSIS_ROOT_REL}`"))
display(summarize_processes(process_catalog))

selected_process_path = Path(PROCESS_DIR) if PROCESS_DIR is not None else Path(process_catalog.loc[PROCESS_INDEX, "process_dir"])
print(f"Diretório raiz da análise: {ANALYSIS_ROOT}")
print(f"Processo selecionado: {selected_process_path}")


In [ ]:
bundle = load_process_bundle(selected_process_path, PROJECT_ROOT)
summary_df = pd.DataFrame(bundle["summary"].items(), columns=["field", "value"])

display(Markdown("## Resumo do processo selecionado"))
display(summary_df)

saved_best_origin = bundle["summary"]["saved_best_origin"]
if saved_best_origin == "train":
    display(Markdown(
        "> `best_prompt.json` corresponde ao melhor prompt de **treino**. O melhor prompt de **validação** ficou em outra época."
    ))
elif saved_best_origin == "train_and_validation":
    display(Markdown(
        "> `best_prompt.json` coincide com o melhor prompt de **treino** e de **validação**."
    ))
elif saved_best_origin == "validation":
    display(Markdown(
        "> `best_prompt.json` coincide com o melhor prompt de **validação**."
    ))
else:
    display(Markdown(
        "> Não foi possível determinar se `best_prompt.json` coincide com o melhor prompt de treino ou de validação."
    ))

display(Markdown("## Comparação entre prompt base, melhor salvo, melhor treino e melhor validação"))
display(bundle["prompt_df"][["prompt_role", "epoch", "source_metric", "score", "prompt_preview"]])


In [ ]:
display(Markdown("## Texto completo dos principais prompts"))
for row in bundle["prompt_df"].itertuples(index=False):
    if not isinstance(row.prompt_text, str) or not row.prompt_text.strip():
        continue
    score_text = "-" if row.score is None or pd.isna(row.score) else f"{row.score:.6f}"
    epoch_text = "-" if row.epoch is None or pd.isna(row.epoch) else int(row.epoch)
    display(Markdown(f"### {row.prompt_role} | epoch={epoch_text} | score={score_text}"))
    print(row.prompt_text)
    print()


In [ ]:
epochs_df = bundle["epochs_df"].copy()
objective_metric = bundle["summary"]["objective_metric"]

epochs_df = ensure_balance_metrics(epochs_df)

if epochs_df.empty:
    display(Markdown("## Gráficos"))
    display(Markdown("> Nenhuma época foi registrada para este processo."))
else:
    metric_columns = preferred_metric_order(
        sorted(
            {
                column.replace("train_score_", "")
                for column in epochs_df.columns
                if column.startswith("train_score_")
            }
        )
    )
    all_metric_names = []
    for metric_name in [objective_metric, *metric_columns]:
        if metric_name not in all_metric_names:
            all_metric_names.append(metric_name)

    metric_colors = {
        metric_name: metric_color(metric_name)
        for metric_name in all_metric_names
    }
    epoch_positions = epochs_df["epoch"].tolist()
    epoch_labels = [epoch + 1 for epoch in epoch_positions]
    best_train_epochs = epochs_df.loc[epochs_df["is_best_train_epoch"], "epoch"].tolist()
    best_val_epochs = epochs_df.loc[epochs_df["is_best_val_epoch"], "epoch"].tolist()
    shared_best_epochs = sorted(set(best_train_epochs) & set(best_val_epochs))
    train_only_epochs = [epoch for epoch in best_train_epochs if epoch not in shared_best_epochs]
    val_only_epochs = [epoch for epoch in best_val_epochs if epoch not in shared_best_epochs]

    display(Markdown(f"## Gráfico focado na métrica objetivo ({objective_metric})"))
    best_train_text = ", ".join(str(epoch + 1) for epoch in train_only_epochs) if train_only_epochs else "nenhuma"
    best_val_text = ", ".join(str(epoch + 1) for epoch in val_only_epochs) if val_only_epochs else "nenhuma"
    shared_best_text = ", ".join(str(epoch + 1) for epoch in shared_best_epochs) if shared_best_epochs else "nenhuma"
    display(Markdown(
        f"Linhas verticais: roxo = melhor época só em treino ({best_train_text}); rosa = melhor época só em validação ({best_val_text}); cinza = mesma época foi a melhor em treino e validação ({shared_best_text})."
    ))
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(
        epochs_df["epoch"],
        epochs_df["train_metric"],
        marker="o",
        linewidth=2,
        linestyle="-",
        color=metric_colors[objective_metric],
        label=f"train::{objective_metric}",
    )
    if epochs_df["val_metric"].notna().any():
        ax.plot(
            epochs_df["epoch"],
            epochs_df["val_metric"],
            marker="o",
            linewidth=2,
            linestyle="--",
            color=metric_colors[objective_metric],
            label=f"val::{objective_metric}",
        )

    for epoch in train_only_epochs:
        ax.axvline(epoch, color="#7b2cbf", linestyle=":", linewidth=2, alpha=0.9)
    for epoch in val_only_epochs:
        ax.axvline(epoch, color="#e75480", linestyle=":", linewidth=2, alpha=0.9)
    for epoch in shared_best_epochs:
        ax.axvline(epoch, color="#6c757d", linestyle="-", linewidth=2.2, alpha=0.95)

    ax.set_title(f"Evolução da métrica objetivo ({objective_metric})")
    ax.set_xlabel("época")
    ax.set_ylabel("score")
    ax.set_xticks(epoch_positions)
    ax.set_xticklabels(epoch_labels)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

    display(Markdown("## Gráfico com todas as métricas"))
    if metric_columns:
        if "geom_mean" in metric_columns or "mean_balance" in metric_columns:
            display(Markdown(
                "`geom_mean = sqrt(SEP * ETD)` e "
                "`mean_balance = ((SEP + ETD) / 2) * (1 - abs(SEP - ETD))`."
            ))
        fig, ax = plt.subplots(figsize=(14, 6))
        for metric_name in metric_columns:
            train_col = f"train_score_{metric_name}"
            val_col = f"val_score_{metric_name}"
            line_color = metric_colors[metric_name]
            ax.plot(
                epochs_df["epoch"],
                epochs_df[train_col],
                marker="o",
                linewidth=1.8,
                linestyle="-",
                color=line_color,
                label=f"train::{metric_name}",
            )
            if val_col in epochs_df.columns and epochs_df[val_col].notna().any():
                ax.plot(
                    epochs_df["epoch"],
                    epochs_df[val_col],
                    marker="o",
                    linewidth=1.8,
                    linestyle="--",
                    color=line_color,
                    label=f"val::{metric_name}",
                )
        ax.set_title("Evolução de todas as métricas por época")
        ax.set_xlabel("época")
        ax.set_ylabel("score")
        ax.set_xticks(epoch_positions)
        ax.set_xticklabels(epoch_labels)
        ax.legend(loc="best", ncol=2)
        plt.tight_layout()
        plt.show()
    else:
        display(Markdown("Nenhuma métrica detalhada foi encontrada para este processo."))


In [ ]:
display(Markdown("## Tabela por época"))
if epochs_df.empty:
    display(Markdown("> Nenhuma época foi registrada para este processo."))
else:
    epoch_columns = [
        "epoch",
        "generated_new_prompt",
        "train_metric",
        "val_metric",
        "val_improvement_vs_prev",
        "is_best_train_epoch",
        "is_best_val_epoch",
        "is_saved_best_epoch",
        "time_spent_instruction",
        "time_spent_train_eval",
        "time_spent_val_eval",
        "mmr_selected_reference_epochs",
        "prompt_preview",
    ]
    score_columns = []
    for metric_name in preferred_metric_order(
        sorted(
            {
                column.replace("train_score_", "").replace("val_score_", "")
                for column in epochs_df.columns
                if column.startswith("train_score_") or column.startswith("val_score_")
            }
        )
    ):
        for column in (f"train_score_{metric_name}", f"val_score_{metric_name}"):
            if column in epochs_df.columns:
                score_columns.append(column)
    display(epochs_df[epoch_columns + score_columns])
